In [12]:
import polars as pl
import json
import unicodedata
import os

In [13]:
from config import DADOS_DIR, DUMP_DIR

In [14]:
pl.Config.set_tbl_cols(-1) # mostrar todas as COLUNAS do DataFrame

polars.config.Config

In [15]:
def criar_dicionario():
    if os.path.exists('dicionario_variaveis.json'):
        print('dicionário já existe!')
        return

    # pega dos dados do dicionário de variáveis do Enem 2025 em formato de DataFrame
    caminho_dict_excel = f'{DUMP_DIR}/microdados_enem_2025/DICIONÁRIO/Dicionário_Microdados_Enem_2025.xlsx'
    df_dict = pl.DataFrame(pl.read_excel(caminho_dict_excel, sheet_name="PARTICIPANTES_2025"))

    df_dict.columns = ['NOME_VARIAVEL', 'DESCRICAO', 'CATEGORIA', 'DESCRICAO_CATEGORIA', 'TAMANHO', 'TIPO']
    df_dict = df_dict.slice(2)

    df_dict = df_dict.filter(
        ~(pl.col('NOME_VARIAVEL').is_not_null() & pl.col('DESCRICAO').is_null())
    )

    # transforma o DataFrame em um Json para facilitar a leitura e manipulação dos dados
    df_categorias = df_dict.with_columns(
        pl.col('NOME_VARIAVEL').forward_fill()
    ).filter(
        pl.col('CATEGORIA').is_not_null() & pl.col('DESCRICAO_CATEGORIA').is_not_null()
    ).select(['NOME_VARIAVEL', 'CATEGORIA', 'DESCRICAO_CATEGORIA'])

    dicionario_variaveis = {}
    for nome_variavel, categoria, descricao_categoria in df_categorias.iter_rows():
        dicionario_variaveis.setdefault(nome_variavel, {})[categoria] = descricao_categoria

    with open('dicionario_variaveis.json', 'w', encoding='utf-8') as f:
        json.dump(dicionario_variaveis, f, ensure_ascii=False, indent=2)

    print('dicionário criado com sucesso!')

criar_dicionario()

dicionário já existe!


In [16]:
def datraframe_microdados(ano:int, tipo_dado:str, teste:bool = False) -> pl.DataFrame:
    CAMINHO_CSV = f'{DADOS_DIR}/dados_{ano}/{tipo_dado}_{ano}.csv'

    if teste:
        df = pl.DataFrame(pl.read_csv(CAMINHO_CSV, separator=";", encoding="latin1", n_rows=50))
    else:
        df = pl.DataFrame(pl.read_csv(CAMINHO_CSV, separator=";", encoding="latin1"))
    
    return df

In [17]:
class Candidato:
    """
    Classe para conter todos os dados de um candidato, os dados já serão tratados para estarem a par com as métricas utilizadas
    pelos microdados.
    """

    with open('dicionario_variaveis.json', 'r', encoding='utf-8') as f:
        DICIONARIO = json.load(f)

    def __init__(self, idade:int, sexo:str, raca:str, nacionalidade:str, st_ens_med:str, ano_concluiu:str, treineiro:bool):
        (   self.f_etaria,
            self.sexo,
            self.raca,
            self.nacionalidade,
            self.st_ens_med,
            self.ano_concluiu,
            self.treineiro,
        ) = self.tratar_candidato(idade, sexo, raca, nacionalidade, st_ens_med, ano_concluiu, treineiro)

    @staticmethod
    def _normalizar(texto:str) -> str:
        texto = unicodedata.normalize('NFKD', str(texto)).encode('ascii', 'ignore').decode('ascii')
        return texto.strip().lower()

    @classmethod
    def _buscar_codigo(cls, nome_variavel:str, descricao:str) -> str:
        """Busca o código correspondente à descrição de uma variável no dicionário."""
        categorias = cls.DICIONARIO[nome_variavel]

        # já é um código válido
        if str(descricao) in categorias:
            return str(descricao)

        alvo = cls._normalizar(descricao)
        for codigo, desc in categorias.items():
            if cls._normalizar(desc) == alvo:
                return codigo

        raise ValueError(f"Valor '{descricao}' não encontrado em {nome_variavel}")

    @staticmethod
    def _calcular_faixa_etaria(idade:int) -> str:
        """Converte a idade em anos no código de TP_FAIXA_ETARIA."""
        if idade < 17:
            return '1'
        if idade > 70:
            return '20'
        if idade <= 24:
            # 17 -> 2, 18 -> 3, ..., 24 -> 9
            return str(idade - 15)

        faixas = [
            (25, 25, '10'),
            (26, 30, '11'),
            (31, 35, '12'),
            (36, 40, '13'),
            (41, 45, '14'),
            (46, 50, '15'),
            (51, 55, '16'),
            (56, 60, '17'),
            (61, 65, '18'),
            (66, 70, '19'),
        ]
        for minimo, maximo, codigo in faixas:
            if minimo <= idade <= maximo:
                return codigo

        raise ValueError(f"Idade inválida: {idade}")

    def tratar_candidato(self, idade:int, sexo:str, raca:str, nacionalidade:str, st_ens_med:str, ano_concluiu:str, treineiro:bool):
        _f_etaria = self._calcular_faixa_etaria(idade)
        _sexo = self._buscar_codigo('TP_SEXO', sexo)
        _raca = self._buscar_codigo('TP_COR_RACA', raca)
        _nacionalidade = self._buscar_codigo('TP_NACIONALIDADE', nacionalidade)
        _st_ens_med = self._buscar_codigo('TP_ST_CONCLUSAO', st_ens_med)
        _ano_concluiu = self._buscar_codigo('TP_ANO_CONCLUIU', ano_concluiu)
        _treineiro = 1 if treineiro else 0

        return _f_etaria, _sexo, _raca, _nacionalidade, _st_ens_med, _ano_concluiu, _treineiro

In [18]:
def filtrar_candidato(df_raw: pl.DataFrame, candidato: Candidato):
    df = datraframe_microdados(2025, "PARTICIPANTES", teste=True)
    df_filtrado = df.filter(
        (pl.col("TP_FAIXA_ETARIA") == int(candidato.f_etaria)) &
        (pl.col("TP_SEXO") == candidato.sexo) &
        (pl.col("TP_COR_RACA") == int(candidato.raca)) &
        (pl.col("TP_NACIONALIDADE") == int(candidato.nacionalidade)) &
        (pl.col("TP_ST_CONCLUSAO") == int(candidato.st_ens_med)) &
        (pl.col("TP_ANO_CONCLUIU") == int(candidato.ano_concluiu)) &
        (pl.col("IN_TREINEIRO") == candidato.treineiro)
    )
    return df_filtrado

---
### FLUXO PRINCIPAL

In [ ]:
# pega os dados do ano
df = datraframe_microdados(2025, "PARTICIPANTES", teste=True)
print(len(df))

# filtra o dataframe pelas características do candidato
df_filtrado = filtrar_candidato(df, Candidato(idade=18, sexo='Feminino', raca='Branca',
                                               nacionalidade='Brasileiro(a)',
                                               st_ens_med='Estou cursando e concluirei o Ensino Médio em 2025',
                                               ano_concluiu='Não informado', treineiro=False))
print(len(df_filtrado))

df_filtrado.head()

50
2


NU_INSCRICAO,NU_ANO,TP_FAIXA_ETARIA,TP_SEXO,TP_ESTADO_CIVIL,TP_COR_RACA,TP_NACIONALIDADE,TP_ST_CONCLUSAO,TP_ANO_CONCLUIU,TP_ENSINO,IN_TREINEIRO,CO_MUNICIPIO_PROVA,NO_MUNICIPIO_PROVA,CO_UF_PROVA,SG_UF_PROVA,Q001,Q002,Q003,Q004,Q005,Q006,Q007,Q008,Q009,Q010,Q011,Q012,Q013,Q014,Q015,Q016,Q017,Q018,Q019,Q020,Q021,Q022,Q023
i64,i64,i64,str,i64,i64,i64,i64,i64,i64,i64,i64,str,i64,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
210066506243,2025,3,"""F""",1,1,1,2,0,1,0,3550308,"""São Paulo""",35,"""SP""","""E""","""E""","""D""","""D""",4,"""A""","""F""","""A""","""C""","""D""","""C""","""A""","""B""","""A""","""B""","""B""","""B""","""B""","""A""","""B""","""B""","""B""","""D"""
210066506253,2025,3,"""F""",1,1,1,2,0,1,0,4305108,"""Caxias do Sul""",43,"""RS""","""F""","""E""","""E""","""D""",3,"""A""","""H""","""A""","""C""","""C""","""C""","""A""","""B""","""A""","""B""","""B""","""B""","""B""","""B""","""B""","""B""","""D""","""B"""
